# GurIA Nails dataset check

Este notebook localiza o dataset anexado, valida os 50 pares revisados e mostra uma amostra visual.

In [ ]:
from pathlib import Path
import csv
from PIL import Image

manifest_files = list(Path('/kaggle/input').rglob('review_manifest.csv'))
assert manifest_files, 'review_manifest.csv nao encontrado; anexe o dataset ao notebook.'
dataset_root = manifest_files[0].parent
train_root = dataset_root / 'train'
rgba_root = dataset_root / 'transparent-originals'
dataset_root

In [ ]:
images = sorted(train_root.glob('*.png'))
captions = sorted(train_root.glob('*.txt'))
rgba_images = sorted(rgba_root.glob('*.png'))
with (dataset_root / 'review_manifest.csv').open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
approved = [row for row in rows if row['status'] == 'approved']

assert len(images) == 50, len(images)
assert len(captions) == 50, len(captions)
assert len(rgba_images) == 50, len(rgba_images)
assert len(rows) == 140, len(rows)
assert len(approved) == 50, len(approved)

for image_path in images:
    caption_path = image_path.with_suffix('.txt')
    assert caption_path.exists(), caption_path
    caption = caption_path.read_text(encoding='utf-8-sig').strip()
    assert caption.startswith('naildecalstyle, '), caption_path.name
    assert caption.lower().count('naildecalstyle') == 1, caption_path.name
    with Image.open(image_path) as image:
        assert image.mode == 'RGB', (image_path.name, image.mode)
        effective_size = image.size
    with Image.open(rgba_root / image_path.name) as rgba:
        assert rgba.mode == 'RGBA', (image_path.name, rgba.mode)
        assert rgba.getchannel('A').getextrema()[0] < 255, image_path.name
        assert rgba.size == effective_size, (image_path.name, rgba.size, effective_size)

print('PASS: 50 pares RGB+TXT, 50 originais RGBA, resolução original e trigger coerente.')

In [ ]:
import matplotlib.pyplot as plt

sample = images[::2][:20]
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for axis, image_path in zip(axes.flat, sample):
    with Image.open(image_path) as image:
        axis.imshow(image)
    axis.set_title(image_path.stem, fontsize=8)
    axis.axis('off')
plt.tight_layout()

## Próximo passo

Use o pacote completo no RunPod, faça primeiro o perfil `smoke` e só então o treino `quality`.